# Laboratorio práctico · Semana 6 | Ciencia de datos con Python
**Formación SENA · Caso integrador: seguimiento académico ficticio**  
**Tema:** Agrupación y agregación, KPI, tendencia central, dispersión y participación.  
**Continuidad:** usa el CSV derivado de Semana 5, sin sobreescribirlo. **Fecha de corte de la fuente:** 18/10/2026.

**Importante:** los umbrales de 3.5 y 80% son únicamente supuestos didácticos, NO criterios oficiales del SENA. Los registros y nombres son ficticios. No interprete indicadores descriptivos como diagnósticos personales ni riesgo real de deserción.

## Antes de comenzar
1. Conserve la estructura: `data/processed/`, `notebooks/`, `outputs/`, `docs/`.
2. Copie `semana5_seguimiento_transformado_para_semana6.csv` a `data/processed/semana5_seguimiento_transformado.csv`.
3. Active el entorno de la Semana 5 e instale si hace falta: `python -m pip install pandas numpy jupyter ipykernel`.
4. Abra el notebook desde `notebooks/` o desde la raíz del proyecto y ejecútelo en orden con *Restart & Run All*.
5. En cada pregunta indique **universo, numerador, denominador, unidad, fecha de corte y limitaciones**.

## 1. Recuperación de Semana 5: fuente y calidad
Observe el universo original. No confunda **filas**, **documentos únicos** y **registros aptos**; un documento está repetido con calificaciones contradictorias.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FUENTE = RAIZ / "data" / "processed" / "semana5_seguimiento_transformado.csv"
SALIDA = RAIZ / "outputs"
SALIDA.mkdir(parents=True, exist_ok=True)
assert FUENTE.is_file(), f"Coloque el CSV de S5 en: {FUENTE}"

df = pd.read_csv(FUENTE, dtype={"documento": "string", "ficha": "string"}, encoding="utf-8-sig")
for col in ["fecha_actualizacion", "fecha_corte"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
for col in ["nota_python", "nota_datos", "asistencia", "promedio_dos_notas"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
for col in ["conflicto_identidad", "fecha_fuera_corte"]:
    assert df[col].notna().all(), f"Revise bandera: {col}"
    df[col] = df[col].astype(bool)
print("Registros heredados:", len(df))
print("Columnas:", len(df.columns))
print("Corte:", df["fecha_corte"].dt.date.dropna().unique().tolist())

Registros heredados: 19
Columnas: 27
Corte: [datetime.date(2026, 10, 18)]


In [2]:
# Regla heredada de S5: no resolver conflictos por intuición.
base = df.loc[
    ~df["conflicto_identidad"] & ~df["fecha_fuera_corte"] &
    df["fecha_actualizacion"].notna() &
    df["documento"].notna() &
    df["promedio_dos_notas"].notna() &
    df["asistencia"].notna()
].copy()
print("Filas fuente:", len(df), "| Base apta:", len(base))
print("Documentos distintos en base:", base["documento"].nunique())
assert len(df) == 19 and len(base) == 11  # control del dataset entregado
assert base["documento"].is_unique
assert not base["fecha_fuera_corte"].any()
assert not base["conflicto_identidad"].any()

Filas fuente: 19 | Base apta: 11
Documentos distintos en base: 11


**Pregunta de análisis:** ¿Por qué una media calculada sobre 19 filas no equivale necesariamente a una media de 19 aprendices? ¿Qué registros quedaron fuera y por qué? No sume causas de exclusión como si fueran categorías mutuamente excluyentes.

## 2. Patrón Split–Apply–Combine
1. **Split:** agrupe las filas por una variable, como `programa_limpio`.
2. **Apply:** calcule el número de registros o promedio dentro de cada grupo.
3. **Combine:** obtenga una tabla resumida de resultados.

No confunda `size()` (cuenta filas, incluso con nulos en otras columnas) con `count()` (cuenta valores NO nulos de una variable).

In [3]:
# Ejemplo simple: media por programa en los 11 registros aptos.
media_programa = base.groupby("programa_limpio", dropna=False)[
    "promedio_dos_notas"
].mean()
print(media_programa)

# Conteo por programa: el total debe coincidir con la base apta.
conteo_programa = base.groupby("programa_limpio", dropna=False).size()
print(conteo_programa)
assert int(conteo_programa.sum()) == len(base)

programa_limpio
ADSO                           3.965
Ciencia de datos con Python    4.250
Name: promedio_dos_notas, dtype: float64
programa_limpio
ADSO                           10
Ciencia de datos con Python     1
dtype: int64


## 3. `agg()`: varios KPI por segmento
Las medidas de dispersión ayudan a observar si los valores son parecidos o muy distintos. `std()` en Pandas usa desviación estándar **muestral** (`ddof=1`) de forma predeterminada; en grupos de un único registro no se puede estimar esa dispersión con dicha fórmula. Evite comparaciones contundentes entre grupos de tamaños muy diferentes.

In [4]:
resumen_programa = (
    base.groupby("programa_limpio", dropna=False)
    .agg(
        registros=("documento", "size"),
        documentos_unicos=("documento", "nunique"),
        media_notas=("promedio_dos_notas", "mean"),
        mediana_notas=("promedio_dos_notas", "median"),
        desviacion_notas=("promedio_dos_notas", "std"),
        minimo_notas=("promedio_dos_notas", "min"),
        maximo_notas=("promedio_dos_notas", "max"),
        media_asistencia=("asistencia", "mean")
    ).reset_index()
)
resumen_programa["rango_notas"] = (resumen_programa["maximo_notas"] -
                                   resumen_programa["minimo_notas"])
print(resumen_programa.to_string(index=False))
assert int(resumen_programa["registros"].sum()) == len(base)

            programa_limpio  registros  documentos_unicos  media_notas  mediana_notas  desviacion_notas  minimo_notas  maximo_notas  media_asistencia  rango_notas
                       ADSO         10                 10        3.965           3.90          0.694442          3.00          4.95              84.7         1.95
Ciencia de datos con Python          1                  1        4.250           4.25               NaN          4.25          4.25              93.0         0.00


**Lectura crítica:** un promedio describe la base apta, no a toda la ficha. La mediana puede ser más robusta ante valores extremos. Si un grupo solo tiene 1 registro, la desviación estándar muestral es `NaN` y no debe rellenarse con 0 para afirmar que «no varía».

## 4. `value_counts()`: distribución y participación
Primero cuente categorías. Después obtenga porcentajes sobre el mismo universo. `normalize=True` devuelve proporciones; multiplique por 100 si desea porcentaje. Use `dropna=False` cuando sea importante mostrar los faltantes.

In [5]:
frecuencias = base["prioridad_revision"].value_counts(dropna=False)
participaciones = (base["prioridad_revision"].value_counts(
    normalize=True, dropna=False) * 100
)
print("Conteos:\n", frecuencias)
print("Participación (%):\n", participaciones.round(2))
assert int(frecuencias.sum()) == len(base)
assert np.isclose(participaciones.sum(), 100)

Conteos:
 prioridad_revision
Seguimiento habitual    8
Prioridad alta          2
Prioridad preventiva    1
Name: count, dtype: int64
Participación (%):
 prioridad_revision
Seguimiento habitual    72.73
Prioridad alta          18.18
Prioridad preventiva     9.09
Name: proportion, dtype: float64


## 5. `crosstab()`: frecuencia cruzada por dos categorías
La tabla de doble entrada permite comparar distribuciones, por ejemplo `programa_limpio` × `prioridad_revision`. **Cuidado:** en `normalize="index"` cada fila suma 100%, pero eso NO indica participación de cada grupo en todo el universo.

In [6]:
tabla_cruzada = pd.crosstab(
    base["programa_limpio"],
    base["prioridad_revision"],
    margins=True, margins_name="Total"
)
por_filas = pd.crosstab(
    base["programa_limpio"],
    base["prioridad_revision"],
    normalize="index"
).mul(100).round(1)
print("Conteos cruzados:\n", tabla_cruzada)
print("Porcentaje dentro de cada programa:\n", por_filas)
assert int(tabla_cruzada.loc["Total", "Total"]) == len(base)

Conteos cruzados:
 prioridad_revision           Prioridad alta  Prioridad preventiva  \
programa_limpio                                                     
ADSO                                      2                     1   
Ciencia de datos con Python               0                     0   
Total                                     2                     1   

prioridad_revision           Seguimiento habitual  Total  
programa_limpio                                           
ADSO                                            7     10  
Ciencia de datos con Python                     1      1  
Total                                           8     11  
Porcentaje dentro de cada programa:
 prioridad_revision           Prioridad alta  Prioridad preventiva  \
programa_limpio                                                     
ADSO                                   20.0                  10.0   
Ciencia de datos con Python             0.0                   0.0   

prioridad_revision        

## 6. `pivot_table()`: indicadores en filas y columnas
Obtenga la media de dos notas por programa y ciudad. Los cruces sin observaciones quedan en `NaN`: **no** sustituyas esos faltantes por 0, porque «no existe observación» no es una calificación de cero. Use `ciudad_reporte` para conservar explícitamente la categoría «No reportada».

In [7]:
tabla_pivote = pd.pivot_table(
    base,
    index="programa_limpio", columns="ciudad_reporte",
    values="promedio_dos_notas", aggfunc="mean"
)
conteos_ciudad = pd.crosstab(
    base["programa_limpio"], base["ciudad_reporte"]
)
print("Media por programa/ciudad:\n", tabla_pivote.round(2))
print("Conteos de registros por ciudad:\n", conteos_ciudad)
assert int(conteos_ciudad.to_numpy().sum()) == len(base)

Media por programa/ciudad:
 ciudad_reporte               Bello  Envigado  Itagüí  Medellín  No reportada
programa_limpio                                                             
ADSO                          3.87      4.95    3.92      3.81           NaN
Ciencia de datos con Python    NaN       NaN     NaN       NaN          4.25
Conteos de registros por ciudad:
 ciudad_reporte               Bello  Envigado  Itagüí  Medellín  No reportada
programa_limpio                                                             
ADSO                             3         1       2         4             0
Ciencia de datos con Python      0         0       0         0             1


## 7. KPI institucionales didácticos: ficha técnica mínima
Cada KPI debe declarar: **nombre, objetivo, fórmula, numerador, denominador, unidad, fecha de corte, universo, responsable de validación y limitación**. Evite etiquetar a personas con un resultado agregado y no asuma causalidad. La referencia a un umbral académico es solo para practicar.

In [8]:
total = len(df)
aptos = len(base)
conteo_prioridades = base["prioridad_revision"].value_counts()
altas = int(conteo_prioridades.get("Prioridad alta", 0))
kpis = pd.DataFrame([
    {"kpi": "Cobertura analítica", "valor": 100 * aptos / total,
     "unidad": "% de filas", "numerador": aptos,
     "denominador": total, "universo": "19 filas recibidas"},
    {"kpi": "Media de promedio de notas", "valor": base["promedio_dos_notas"].mean(),
     "unidad": "puntos (0-5)", "numerador": np.nan,
     "denominador": aptos, "universo": "Solo registros aptos"},
    {"kpi": "Participación prioridad alta", "valor": 100 * altas / aptos,
     "unidad": "% de base apta", "numerador": altas,
     "denominador": aptos, "universo": "Solo registros aptos"},
])
print(kpis.round(2).to_string(index=False))
assert kpis.loc[0, "denominador"] == total
assert 0 <= kpis.loc[0, "valor"] <= 100

                         kpi  valor         unidad  numerador  denominador             universo
         Cobertura analítica  57.89     % de filas       11.0           19   19 filas recibidas
  Media de promedio de notas   3.99   puntos (0-5)        NaN           11 Solo registros aptos
Participación prioridad alta  18.18 % de base apta        2.0           11 Solo registros aptos


## 8. Tendencia central, dispersión y participación
Analice una media, una mediana y la desviación estándar **muestral** de `promedio_dos_notas`. Pregunte: ¿Cuántos registros componen la muestra? ¿Qué información queda fuera? ¿Qué cambia si el universo se modifica?

In [9]:
medidas = pd.Series({
    "n": len(base),
    "media": base["promedio_dos_notas"].mean(),
    "mediana": base["promedio_dos_notas"].median(),
    "desviacion_muestral": base["promedio_dos_notas"].std(ddof=1),
    "minimo": base["promedio_dos_notas"].min(),
    "maximo": base["promedio_dos_notas"].max(),
    "rango": (base["promedio_dos_notas"].max() -
              base["promedio_dos_notas"].min())
})
print(medidas.round(3))
participacion_programa = (
    base["programa_limpio"].value_counts(dropna=False,
    normalize=True) * 100
)
print("Participación por programa (%):\n", participacion_programa.round(2))

n                      11.000
media                   3.991
mediana                 4.100
desviacion_muestral     0.664
minimo                  3.000
maximo                  4.950
rango                   1.950
dtype: float64
Participación por programa (%):
 programa_limpio
ADSO                           90.91
Ciencia de datos con Python     9.09
Name: proportion, dtype: float64


## 9. Control cruzado, exportación y trazabilidad
Compare cada resumen con la cantidad de filas aptas. Guarde CSV de agregados y KPI sin modificar el dataset recibido. Registre en el README la fecha de corte, los supuestos y la limitación de muestra.

In [10]:
# Invariantes del laboratorio
assert len(df) == 19 and len(base) == 11
assert resumen_programa["registros"].sum() == len(base)
assert conteos_ciudad.to_numpy().sum() == len(base)
assert base["documento"].nunique() == len(base)
assert (base["fecha_actualizacion"] <= base["fecha_corte"]).all()

resumen_programa.to_csv(SALIDA / "semana6_agrupacion_programa.csv", index=False, encoding="utf-8-sig")
tabla_cruzada.to_csv(SALIDA / "semana6_crosstab.csv", encoding="utf-8-sig")
tabla_pivote.to_csv(SALIDA / "semana6_tabla_dinamica.csv", encoding="utf-8-sig")
kpis.to_csv(SALIDA / "semana6_kpis.csv", index=False, encoding="utf-8-sig")
print("Productos guardados en:", SALIDA)

Productos guardados en: c:\7_trimestre\CienciaDatos\ciencia-datos-python\outputs


## 10. Retos autónomos — NO incluyen solución
**Reto A:** calcule `groupby("ciudad_reporte").agg(...)` para contar registros, media y mediana de notas. Comente cómo interpretar «No reportada».

**Reto B:** construya una tabla cruzada `ciudad_reporte × prioridad_revision` y una participación por fila. Explique ambos denominadores.

**Reto C:** registre un KPI nuevo (objetivo, fórmula, numerador, denominador, universo, corte y limitación), con comprobación `assert` y exportación.

**Reto D:** compare media y mediana por dos segmentos: explique por qué una diferencia descriptiva no prueba causa ni autoriza decisiones sobre un aprendiz.

**Entrega:** notebook ejecutable, cuatro CSV exportados, ficha técnica de 3 KPI y conclusión de 150–200 palabras. **Sin datos personales reales**.

In [11]:
# RETO A - Agrupación de registros académicos por ciudad

reto_a = (
    base.groupby("ciudad_reporte", dropna=False)
    .agg(
        registros=("documento", "size"),
        media_notas=("promedio_dos_notas", "mean"),
        mediana_notas=("promedio_dos_notas", "median")
    )
    .reset_index()
)

# Mostrar resultados con dos decimales
print("RETO A - RESUMEN POR CIUDAD")
print(reto_a.round(2).to_string(index=False))

# Comprobar que se incluyeron los 11 registros aptos
assert reto_a["registros"].sum() == len(base)

RETO A - RESUMEN POR CIUDAD
ciudad_reporte  registros  media_notas  mediana_notas
         Bello          3         3.87           3.70
      Envigado          1         4.95           4.95
        Itagüí          2         3.92           3.92
      Medellín          4         3.81           3.78
  No reportada          1         4.25           4.25


Los registros aptos se agruparon por ciudad para calcular la cantidad de observaciones, la media y la mediana de las calificaciones. La categoría «No reportada» se conserva porque representa registros sin información de ciudad. No debe interpretarse como una ubicación geográfica ni eliminarse automáticamente, ya que esto modificaría el universo analizado.

In [12]:
# RETO B - Tabla cruzada por ciudad y prioridad

reto_b = pd.crosstab(
    base["ciudad_reporte"],
    base["prioridad_revision"],
    margins=True,
    margins_name="Total"
)

# Porcentajes dentro de cada ciudad
reto_b_porcentajes = pd.crosstab(
    base["ciudad_reporte"],
    base["prioridad_revision"],
    normalize="index"
) * 100

print("RETO B - CONTEOS POR CIUDAD Y PRIORIDAD")
print(reto_b)

print("\nRETO B - PORCENTAJES POR CIUDAD")
print(reto_b_porcentajes.round(2))

# Validar el total de registros
assert reto_b.loc["Total", "Total"] == len(base)

# Verificar que los porcentajes por ciudad sumen 100 %
assert np.allclose(reto_b_porcentajes.sum(axis=1), 100)

RETO B - CONTEOS POR CIUDAD Y PRIORIDAD
prioridad_revision  Prioridad alta  Prioridad preventiva  \
ciudad_reporte                                             
Bello                            0                     0   
Envigado                         0                     0   
Itagüí                           1                     0   
Medellín                         1                     1   
No reportada                     0                     0   
Total                            2                     1   

prioridad_revision  Seguimiento habitual  Total  
ciudad_reporte                                   
Bello                                  3      3  
Envigado                               1      1  
Itagüí                                 1      2  
Medellín                               2      4  
No reportada                           1      1  
Total                                  8     11  

RETO B - PORCENTAJES POR CIUDAD
prioridad_revision  Prioridad alta  Prioridad 

### Interpretación del Reto B

La tabla cruzada permite identificar la distribución de las categorías de prioridad de revisión según la ciudad reportada. En Medellín se encontraron cuatro registros aptos: uno de prioridad alta (25 %), uno de prioridad preventiva (25 %) y dos de seguimiento habitual (50 %). En Itagüí, uno de los dos registros corresponde a prioridad alta (50 %).

Los conteos representan cantidades absolutas, mientras que los porcentajes se calculan utilizando como denominador el total de registros aptos de cada ciudad, no los 11 registros de toda la base. Por esta razón, las proporciones de cada ciudad suman 100 %.

Estos resultados son descriptivos y no permiten establecer conclusiones generales sobre el rendimiento académico de los aprendices.

In [13]:
# RETO C - KPI adicional de asistencia

# Definir el umbral didáctico
umbral_asistencia = 80

# Calcular numerador y denominador
total_aptos = len(base)

aptos_asistencia = (
    base["asistencia"] >= umbral_asistencia
).sum()

# Calcular el indicador
kpi_asistencia = (
    aptos_asistencia / total_aptos
) * 100

print("RETO C - KPI DE ASISTENCIA")
print("Registros aptos:", total_aptos)
print("Registros con asistencia >= 80%:", aptos_asistencia)
print("Porcentaje:", round(kpi_asistencia, 2), "%")

# Validaciones
assert total_aptos == 11
assert 0 <= kpi_asistencia <= 100
assert aptos_asistencia <= total_aptos

# Exportar el KPI adicional
kpi_reto_c = pd.DataFrame([{
    "kpi": "Asistencia igual o superior al 80%",
    "valor": round(kpi_asistencia, 2),
    "unidad": "%",
    "numerador": int(aptos_asistencia),
    "denominador": total_aptos,
    "universo": "Registros aptos",
    "fecha_corte": "2026-10-18"
}])

kpi_reto_c.to_csv(
    SALIDA / "semana6_reto_c_kpi_asistencia.csv",
    index=False,
    encoding="utf-8-sig"
)

print("KPI adicional exportado correctamente.")

RETO C - KPI DE ASISTENCIA
Registros aptos: 11
Registros con asistencia >= 80%: 9
Porcentaje: 81.82 %
KPI adicional exportado correctamente.


### Reto C. Ficha técnica del KPI de asistencia

**Nombre:** Porcentaje de registros aptos con asistencia igual o superior al 80 %.

**Objetivo:** Identificar la proporción de registros académicos aptos que cumplen un umbral de asistencia establecido únicamente con fines didácticos.

**Fórmula:** (Registros aptos con asistencia ≥ 80 % / Total de registros aptos) × 100.

**Numerador:** 9 registros.

**Denominador:** 11 registros.

**Resultado:** 81,82 %.

**Unidad de medida:** Porcentaje (%).

**Universo:** Los 11 registros aptos de la base académica ficticia utilizada en la Semana 6.

**Fecha de corte:** 18 de octubre de 2026.

**Responsable de validación:** Aprendiz responsable del análisis, mediante comprobaciones automáticas en Python.

**Limitaciones:** El indicador excluye ocho registros que no cumplen los criterios de calidad establecidos. El umbral del 80 % es un supuesto didáctico y no representa una política oficial del SENA. Los resultados describen únicamente la muestra analizada y no deben utilizarse para evaluar individualmente a los aprendices.

In [14]:
# RETO D - Comparación de dos programas académicos

reto_d = (
    base.groupby("programa_limpio")
    .agg(
        registros=("documento", "size"),
        media=("promedio_dos_notas", "mean"),
        mediana=("promedio_dos_notas", "median")
    )
    .reset_index()
)

print("RETO D - COMPARACIÓN DE PROGRAMAS")
print(reto_d.round(3).to_string(index=False))

# Validaciones
assert reto_d["registros"].sum() == len(base)
assert len(reto_d) == 2
assert reto_d["media"].notna().all()
assert reto_d["mediana"].notna().all()

print("\nComparación realizada correctamente.")

RETO D - COMPARACIÓN DE PROGRAMAS
            programa_limpio  registros  media  mediana
                       ADSO         10  3.965     3.90
Ciencia de datos con Python          1  4.250     4.25

Comparación realizada correctamente.


### Interpretación del Reto D

Se compararon la media y la mediana de las calificaciones de dos programas académicos utilizando únicamente los registros aptos para el análisis.

El programa ADSO presentó una media de 3,965 y una mediana de 3,90, calculadas a partir de diez registros. Por su parte, Ciencia de Datos con Python obtuvo una media y una mediana de 4,25, pero cuenta con una sola observación.

Aunque existe una diferencia descriptiva de 0,285 puntos entre las medias, esta no demuestra que el programa académico sea la causa del resultado. Pueden intervenir otras variables que no fueron evaluadas y los tamaños de los grupos son considerablemente diferentes.

Por lo tanto, estos indicadores no deben utilizarse para establecer conclusiones generales ni tomar decisiones individuales sobre los aprendices.

### Conclusión — Semana 6

Durante el desarrollo de la Semana 6 se aplicaron técnicas de agrupación, agregación y análisis estadístico utilizando Python y la biblioteca Pandas. El trabajo permitió transformar los datos académicos ficticios de la Semana 5 en información organizada mediante tablas, indicadores y comparaciones.

De los 19 registros originales, solamente 11 cumplieron los criterios establecidos para el análisis, lo que demuestra la importancia de verificar la calidad de los datos antes de calcular estadísticas. Se utilizaron funciones como `groupby()`, `agg()`, `value_counts()`, `crosstab()` y `pivot_table()` para analizar las calificaciones, la asistencia y las categorías de seguimiento académico.

Los resultados mostraron una media general de 3,991 y una cobertura analítica del 57,89 %. También se identificó que el 81,82 % de los registros aptos presentó una asistencia igual o superior al 80 %, considerando este umbral únicamente como referencia didáctica.

Finalmente, se comprendió que los indicadores describen únicamente los registros analizados y que las diferencias estadísticas no demuestran relaciones de causalidad. Esta actividad fortaleció mis habilidades para analizar datos, validar resultados y presentar información de manera responsable.

## Fichas técnicas de los KPI — Semana 6

**Contexto general:** Los indicadores se calcularon utilizando registros académicos ficticios, con fecha de corte del 18 de octubre de 2026. Su finalidad es exclusivamente didáctica.

### KPI 1. Cobertura analítica

- **Objetivo:** Determinar qué proporción de los registros recibidos cumple los criterios de calidad establecidos para el análisis.
- **Fórmula:** (Registros aptos / Registros recibidos) × 100.
- **Numerador:** 11 registros aptos.
- **Denominador:** 19 registros recibidos.
- **Resultado:** 57,89 %.
- **Unidad:** Porcentaje (%).
- **Universo:** Los 19 registros originales.
- **Fecha de corte:** 18/10/2026.
- **Responsable de validación:** Aprendiz responsable del análisis, mediante comprobaciones en Python.
- **Limitación:** El indicador refleja la cobertura de los datos disponibles, no el rendimiento académico ni la proporción de aprendices del SENA.

### KPI 2. Media de promedio de notas

- **Objetivo:** Describir la calificación promedio de los registros aptos.
- **Fórmula:** Suma de los promedios de dos notas de cada registro apto / Número de registros aptos.
- **Numerador:** Suma de los promedios individuales de los 11 registros.
- **Denominador:** 11 registros aptos.
- **Resultado:** 3,99 puntos.
- **Unidad:** Puntos en escala de 0 a 5.
- **Universo:** Los 11 registros aptos.
- **Fecha de corte:** 18/10/2026.
- **Responsable de validación:** Aprendiz responsable del análisis, mediante comprobaciones en Python.
- **Limitación:** La media no describe la dispersión de las calificaciones y no representa necesariamente a todos los registros recibidos.

### KPI 3. Participación de prioridad alta

- **Objetivo:** Identificar la proporción de registros aptos clasificados con prioridad alta de revisión.
- **Fórmula:** (Registros aptos con prioridad alta / Total de registros aptos) × 100.
- **Numerador:** 2 registros de prioridad alta.
- **Denominador:** 11 registros aptos.
- **Resultado:** 18,18 %.
- **Unidad:** Porcentaje (%).
- **Universo:** Los 11 registros aptos.
- **Fecha de corte:** 18/10/2026.
- **Responsable de validación:** Aprendiz responsable del análisis, mediante comprobaciones en Python.
- **Limitación:** La categoría de prioridad alta corresponde a reglas didácticas y no representa un diagnóstico académico ni autoriza decisiones individuales sobre los aprendices.